# BirdCLEF+ 2026 — Inference & Submission

Load the trained checkpoint, predict on the soundscapes, write `submission.csv`.

**Note:** `test_soundscapes/` is empty locally — Kaggle reveals the real test audio
only during scored submission. We fall back to a few files from `train_soundscapes/`
so we can actually see the pipeline run end-to-end.


## 1. Imports & device


In [1]:
import os, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
import timm
from scipy.ndimage import convolve1d
from tqdm.auto import tqdm

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)


Device: mps


## 2. Load the checkpoint

The checkpoint contains the model weights **plus** the species ordering and all
mel-spectrogram settings. This guarantees the inference pipeline reproduces
exactly what the model saw during training.


In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data" / "birdclef-2026"
CKPT_DIR     = PROJECT_ROOT / "checkpoints"

# Prefer the latest checkpoint: Phase 1 (model_v2.pt) > full (model.pt) > debug.
for candidate in ["model_v2.pt", "model.pt", "model_debug.pt"]:
    ckpt_path = CKPT_DIR / candidate
    if ckpt_path.exists():
        break
else:
    raise FileNotFoundError(f"No checkpoint found in {CKPT_DIR}. Run 01_train.ipynb first.")

ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

species      = ckpt["species"]
label_to_idx = ckpt["label_to_idx"]
NUM_CLASSES  = ckpt["num_classes"]
BACKBONE     = ckpt["backbone"]
SR           = ckpt["sr"]
N_MELS       = ckpt["n_mels"]
N_FFT        = ckpt["n_fft"]
HOP_LENGTH   = ckpt["hop_length"]
FMIN, FMAX   = ckpt["fmin"], ckpt["fmax"]
CLIP_SEC     = ckpt["clip_sec"]
N_SAMPLES    = SR * CLIP_SEC

print(f"Loaded {ckpt_path.name}")
print(f"Backbone: {BACKBONE}   Classes: {NUM_CLASSES}")


Loaded model_debug.pt
Backbone: efficientnet_b0   Classes: 10


In [3]:
model = timm.create_model(BACKBONE, pretrained=False, in_chans=1, num_classes=NUM_CLASSES)
model.load_state_dict(ckpt["state_dict"])
model = model.to(DEVICE).eval()
print("Model loaded.")


Model loaded.


## 3. Mel spectrogram helper (same settings as training)


In [4]:
def wav_to_mel(wav):
    mel    = librosa.feature.melspectrogram(
        y=wav, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0,
    )
    mel_db = librosa.power_to_db(mel, top_db=80)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_db.astype(np.float32)


## 4. Find the test files

If `test_soundscapes/` is empty (local dev), fall back to 5 files from `train_soundscapes/`.


In [5]:
TEST_DIR  = DATA_DIR / "test_soundscapes"
TRAIN_DIR = DATA_DIR / "train_soundscapes"

test_files = sorted(TEST_DIR.glob("*.ogg")) if TEST_DIR.is_dir() else []
if not test_files:
    test_files = sorted(TRAIN_DIR.glob("*.ogg"))[:5]
    print(f"[fallback] using {len(test_files)} files from train_soundscapes/")
else:
    print(f"Found {len(test_files)} test files.")
test_files[:3]


[fallback] using 5 files from train_soundscapes/


[PosixPath('/Users/harish.r/Documents/kaggle/birdclef2026/data/birdclef-2026/train_soundscapes/BC2026_Train_0001_S08_20250606_030007.ogg'),
 PosixPath('/Users/harish.r/Documents/kaggle/birdclef2026/data/birdclef-2026/train_soundscapes/BC2026_Train_0002_S08_20250607_030007.ogg'),
 PosixPath('/Users/harish.r/Documents/kaggle/birdclef2026/data/birdclef-2026/train_soundscapes/BC2026_Train_0003_S08_20250607_070007.ogg')]

## 5. Slice a 60s soundscape into 12 × 5s chunks

Each soundscape file is exactly 60 seconds. We predict one row per 5-second window —
that's 12 rows per file, matching what `submission.csv` expects.


In [6]:
WINDOW_SEC   = CLIP_SEC          # 5
N_WINDOWS    = 60 // WINDOW_SEC  # 12

def file_to_chunks(path):
    """Load file, pad/truncate to 60s, reshape to (12, 160000)."""
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)

    target = N_WINDOWS * N_SAMPLES   # 12 * 160_000 = 1_920_000 samples (60 s)
    if len(wav) < target:
        wav = np.pad(wav, (0, target - len(wav)))
    else:
        wav = wav[:target]
    return wav.reshape(N_WINDOWS, N_SAMPLES).astype(np.float32)


## 6. Inference loop

For each file:
1. Slice into 12 chunks.
2. Convert each chunk to a mel spectrogram.
3. Stack into a batch of 12, run through the model → 12 × 234 logits.
4. Smooth across the 12 windows in logit space (next cell explains why).
5. Sigmoid → probabilities.

We do everything inside `torch.no_grad()` so PyTorch doesn't track gradients
(faster, less memory).


In [7]:
GAUSSIAN_KERNEL = np.array([0.1, 0.2, 0.4, 0.2, 0.1])  # small bell shape, neighbors get small weight

def smooth_windows(logits, kernel=GAUSSIAN_KERNEL):
    """Weighted-average each window with its 4 neighbors. Damps isolated spikes,
    keeps stretches where neighbors agree. Applied per-species along the time axis."""
    return convolve1d(logits, kernel, axis=0, mode="nearest")


@torch.no_grad()
def predict_file(path):
    chunks = file_to_chunks(path)                                # (12, 160000)
    mels   = np.stack([wav_to_mel(c) for c in chunks])           # (12, 128, T)
    mels   = torch.from_numpy(mels).unsqueeze(1).to(DEVICE)      # (12, 1, 128, T)
    logits = model(mels).cpu().numpy()                           # (12, NUM_CLASSES)
    logits = smooth_windows(logits)                              # smooth in logit space
    probs  = 1.0 / (1.0 + np.exp(-logits))                       # sigmoid -> [0, 1]
    return probs


# Smoke test on one file
probs0 = predict_file(test_files[0])
print(f"output shape: {probs0.shape}   range: [{probs0.min():.3f}, {probs0.max():.3f}]")


output shape: (12, 10)   range: [0.009, 0.987]


## 7. Run inference on all test files


In [8]:
all_rows, all_probs = [], []
t0 = time.time()
for f in tqdm(test_files, desc="infer"):
    basename = f.stem                                            # e.g. BC2026_Test_0001_S05_20250227_010002
    probs    = predict_file(f)                                   # (12, NUM_CLASSES)
    end_secs = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC          # 5, 10, ..., 60
    for k in range(N_WINDOWS):
        all_rows.append(f"{basename}_{end_secs[k]}")
        all_probs.append(probs[k])

all_probs = np.stack(all_probs)
print(f"\n{len(all_rows)} rows in {time.time()-t0:.1f}s")


infer:   0%|          | 0/5 [00:00<?, ?it/s]


60 rows in 0.6s


## 8. Build the submission CSV

The submission file must have:
- **`row_id`** column matching the format `{filename}_{end_second}`.
- **One column per species in the taxonomy**, in the order Kaggle expects (from `sample_submission.csv`).
- Probabilities in [0, 1].

Two complications:
1. We only trained on the species present in our training subset, not all 234. We fill
   missing species columns with a small constant (their per-row average prior).
2. We need to put columns in `sample_submission.csv` order.


In [9]:
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
all_species_in_order = [c for c in sample_sub.columns if c != "row_id"]
print(f"Sample submission expects {len(all_species_in_order)} species columns.")

# Build a (n_rows, num_predicted_species) DataFrame first.
pred_df = pd.DataFrame(all_probs, columns=species)
pred_df.insert(0, "row_id", all_rows)

# Reindex to the full taxonomy order; species we didn't train on become NaN.
sub = pred_df.set_index("row_id").reindex(columns=all_species_in_order)

# Fill unseen-species probabilities with a small constant.
# Using 1/234 (the uniform prior) is a reasonable noncommittal guess.
sub = sub.fillna(1.0 / len(all_species_in_order))
sub = sub.clip(0.0, 1.0).reset_index()

assert list(sub.columns) == list(sample_sub.columns), "Column order mismatch!"
assert sub["row_id"].is_unique
print(f"Submission: {sub.shape[0]} rows × {sub.shape[1]} cols")
sub.head(3)


Sample submission expects 234 species columns.
Submission: 60 rows × 235 cols


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.926784
1,BC2026_Train_0001_S08_20250606_030007_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.925947
2,BC2026_Train_0001_S08_20250606_030007_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.919336


In [10]:
out_path = PROJECT_ROOT / "submission.csv"
sub.to_csv(out_path, index=False)
print(f"Wrote {out_path}   ({out_path.stat().st_size/1024:.1f} KB)")


Wrote /Users/harish.r/Documents/kaggle/birdclef2026/submission.csv   (181.0 KB)


## What just happened

For each 60-second soundscape, you produced 12 rows of 234 probabilities and wrote them in
Kaggle's required format. With the debug checkpoint (10 species) the predictions for the
other 224 species are just uniform priors — not useful for the leaderboard. But the
**pipeline is now end-to-end working**. When you train on the full dataset (`DEBUG = False`),
this same notebook produces real predictions.

## Sanity checks before you trust a real submission

- Each `row_id` is unique → ✓ (we asserted).
- All probabilities are in [0, 1] → ✓ (we clipped).
- No NaN values → no errors expected, but double-check: `sub.isna().any().any()` should be `False`.
- Row count = `12 × number_of_test_files`.
